# Task 2: Empirical Diagnostic of Custom Weight Initializations and Gradient Vanishing

## Objective

The objective of this experiment is to investigate training instability in deep neural networks by studying the effect of different weight initialization strategies on activation distributions and gradient magnitudes.

The experiment compares:

- Uninitialized weights
- Custom Xavier (Glorot) initialization
- Custom Kaiming (He) initialization

A 20-layer Multi-Layer Perceptron (MLP) is used to observe how activation means, activation variances, and weight-gradient magnitudes behave throughout training.

In [ ]:
# ============================================================
# Imports and experiment configuration
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Disable PyTorch autograd
torch.set_grad_enabled(False)

# Device
device = torch.device("cpu")

# Network configuration
INPUT_DIM = 32
HIDDEN_DIM = 64
OUTPUT_DIM = 10
NUM_LAYERS = 20

# Training configuration
EPOCHS = 30
BATCH_SIZE = 64
LEARNING_RATE = 0.01

print("PyTorch version:", torch.__version__)
print("Autograd enabled:", torch.is_grad_enabled())
print("Number of layers:", NUM_LAYERS)

# Dataset and Custom Weight Initialization

A synthetic classification dataset is generated using NumPy/PyTorch.

The network contains 20 hidden/linear layers.

Three initialization modes are considered:

1. **Uninitialized** — weights are generated from a standard normal distribution without variance scaling.
2. **Xavier (Glorot)** — preserves variance more effectively across layers.
3. **Kaiming (He)** — designed for ReLU networks.

All initialization functions are written manually rather than using PyTorch's built-in initialization functions.

In [ ]:
# ============================================================
# Synthetic dataset
# ============================================================

N_SAMPLES = 2000

X = torch.randn(
    N_SAMPLES,
    INPUT_DIM
)

# Generate class labels from a random linear projection
true_W = torch.randn(
    INPUT_DIM,
    OUTPUT_DIM
)

logits = X @ true_W

y = torch.argmax(
    logits,
    dim=1
)

# Train/test split
split = int(0.8 * N_SAMPLES)

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


# ============================================================
# Custom initialization functions
# ============================================================

def uninitialized_weights(n_in, n_out):
    """
    Standard normal initialization without
    variance scaling.
    """

    return torch.randn(
        n_in,
        n_out
    )


def xavier_weights(n_in, n_out):
    """
    Xavier / Glorot uniform initialization.

    Range:
        [-sqrt(6/(n_in+n_out)),
          sqrt(6/(n_in+n_out))]
    """

    limit = np.sqrt(
        6.0 / (n_in + n_out)
    )

    return (
        torch.rand(
            n_in,
            n_out
        ) * 2 * limit
        - limit
    )


def kaiming_weights(n_in, n_out):
    """
    Kaiming / He normal initialization.

    Standard deviation:
        sqrt(2/n_in)
    """

    std = np.sqrt(
        2.0 / n_in
    )

    return (
        torch.randn(
            n_in,
            n_out
        ) * std
    )

# 20-Layer MLP and Diagnostic Tracking Harness

The following implementation creates a 20-layer fully connected MLP.

Each layer stores:

- Weight matrix
- Bias vector
- Activation values
- Weight gradients

Autograd is disabled, so gradients are calculated manually.

For every layer and epoch, the tracking harness records:

### Activation Mean

\[
\mu_l = \frac{1}{N}\sum_i A_i^{[l]}
\]

### Activation Variance

\[
\sigma_l^2 =
\frac{1}{N}\sum_i(A_i-\mu_l)^2
\]

### Gradient Magnitude

\[
G_l =
\frac{1}{N}\sum_i |\nabla W_i^{[l]}|
\]

These statistics allow us to identify activation collapse, unstable activations, vanishing gradients, and exploding gradients.

In [ ]:
# ============================================================
# 20-Layer MLP
# ============================================================

class DeepMLP:

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        num_layers,
        initialization
    ):

        self.num_layers = num_layers
        self.weights = []
        self.biases = []

        dimensions = (
            [input_dim]
            + [hidden_dim] * (num_layers - 1)
            + [output_dim]
        )

        # Create parameters
        for i in range(num_layers):

            n_in = dimensions[i]
            n_out = dimensions[i + 1]

            if initialization == "uninitialized":

                W = uninitialized_weights(
                    n_in,
                    n_out
                )

            elif initialization == "xavier":

                W = xavier_weights(
                    n_in,
                    n_out
                )

            elif initialization == "kaiming":

                W = kaiming_weights(
                    n_in,
                    n_out
                )

            else:

                raise ValueError(
                    "Unknown initialization"
                )

            # Explicitly disable autograd
            W.requires_grad = False

            b = torch.zeros(
                1,
                n_out
            )

            b.requires_grad = False

            self.weights.append(W)
            self.biases.append(b)

    def forward(self, X):

        activations = [X]
        pre_activations = []

        A = X

        for layer in range(
            self.num_layers
        ):

            W = self.weights[layer]
            b = self.biases[layer]

            Z = A @ W + b

            pre_activations.append(Z)

            # ReLU for hidden layers
            if layer < self.num_layers - 1:

                A = torch.relu(Z)

            else:

                A = Z

            activations.append(A)

        return (
            A,
            activations,
            pre_activations
        )


# Test architecture
model = DeepMLP(
    INPUT_DIM,
    HIDDEN_DIM,
    OUTPUT_DIM,
    NUM_LAYERS,
    "xavier"
)

print(
    "Number of weight matrices:",
    len(model.weights)
)

print(
    "First weight shape:",
    model.weights[0].shape
)

print(
    "Last weight shape:",
    model.weights[-1].shape
)

print(
    "All parameters require_grad=False:",
    all(
        not W.requires_grad
        for W in model.weights
    )
)

# Manual Backpropagation and Diagnostic Tracking

Since PyTorch autograd is disabled, the gradients are calculated manually.

For the output layer, Mean Squared Error is used:

\[
L =
\frac{1}{N}
\sum_i
(\hat{Y}_i-Y_i)^2
\]

The derivative is:

\[
\frac{\partial L}{\partial \hat{Y}}
=
\frac{2}{N}
(\hat{Y}-Y)
\]

For each linear layer:

\[
dW = A^T dZ
\]

\[
db = \sum dZ
\]

\[
dA = dZ W^T
\]

For ReLU:

\[
dZ = dA \odot ReLU'(Z)
\]

The diagnostic harness records activation means, activation variances, and gradient magnitudes for every layer at every epoch.

In [ ]:
# ============================================================
# Manual backward propagation
# ============================================================

def manual_backward(
    model,
    Y_true,
    activations,
    pre_activations
):

    gradients = [
        None
        for _ in range(model.num_layers)
    ]

    # Output derivative
    prediction = activations[-1]

    dA = (
        2.0
        * (prediction - Y_true)
        / Y_true.shape[0]
    )

    # Backward through layers
    for layer in range(
        model.num_layers - 1,
        -1,
        -1
    ):

        # ReLU derivative for hidden layers
        if layer < model.num_layers - 1:

            dZ = (
                dA
                * (
                    pre_activations[layer]
                    > 0
                ).float()
            )

        else:

            dZ = dA

        A_previous = activations[layer]

        # Weight gradient
        dW = A_previous.T @ dZ

        # Bias gradient
        db = torch.sum(
            dZ,
            dim=0,
            keepdim=True
        )

        gradients[layer] = (
            dW,
            db
        )

        # Propagate backwards
        if layer > 0:

            dA = (
                dZ
                @ model.weights[layer].T
            )

    return gradients


# ============================================================
# Tracking harness
# ============================================================

def initialize_history(num_layers):

    return {
        "activation_mean": [
            [] for _ in range(num_layers)
        ],

        "activation_variance": [
            [] for _ in range(num_layers)
        ],

        "gradient_mean": [
            [] for _ in range(num_layers)
        ],

        "gradient_variance": [
            [] for _ in range(num_layers)
        ]
    }


def record_statistics(
    history,
    activations,
    gradients
):

    for layer in range(
        len(gradients)
    ):

        activation = activations[
            layer + 1
        ]

        dW = gradients[
            layer
        ][0]

        history[
            "activation_mean"
        ][layer].append(
            activation.mean().item()
        )

        history[
            "activation_variance"
        ][layer].append(
            activation.var().item()
        )

        history[
            "gradient_mean"
        ][layer].append(
            dW.abs().mean().item()
        )

        history[
            "gradient_variance"
        ][layer].append(
            dW.var().item()
        )

# Training and Diagnostic Experiment

Each initialization strategy is trained independently:

- Uninitialized
- Xavier
- Kaiming

The network is intentionally deep with 20 layers.

The experiment tracks how the initialization strategy affects the flow of activations and gradients.

The parameters are manually updated using:

\[
W \leftarrow W-\eta dW
\]

No optimizer and no automatic differentiation are used.

In [ ]:
# ============================================================
# Training experiment
# ============================================================

def run_experiment(
    initialization,
    epochs=30
):

    model = DeepMLP(
        INPUT_DIM,
        HIDDEN_DIM,
        OUTPUT_DIM,
        NUM_LAYERS,
        initialization
    )

    history = initialize_history(
        NUM_LAYERS
    )

    losses = []
    accuracies = []

    for epoch in range(epochs):

        # Mini-batch sampling
        indices = np.random.choice(
            len(X_train),
            BATCH_SIZE,
            replace=False
        )

        X_batch = X_train[indices]
        y_batch = y_train[indices]

        # One-hot targets
        Y_batch = torch.zeros(
            BATCH_SIZE,
            OUTPUT_DIM
        )

        Y_batch[
            torch.arange(BATCH_SIZE),
            y_batch
        ] = 1.0

        # Forward
        output, activations, pre_activations = (
            model.forward(X_batch)
        )

        # Convert logits to softmax probabilities
        probabilities = torch.softmax(
            output,
            dim=1
        )

        # MSE loss
        loss = torch.mean(
            (probabilities - Y_batch) ** 2
        )

        # Manual backward
        gradients = manual_backward(
            model,
            Y_batch,
            activations,
            pre_activations
        )

        # Track statistics
        record_statistics(
            history,
            activations,
            gradients
        )

        # Manual parameter update
        for layer in range(
            model.num_layers
        ):

            dW, db = gradients[layer]

            model.weights[layer] -= (
                LEARNING_RATE * dW
            )

            model.biases[layer] -= (
                LEARNING_RATE * db
            )

        # Accuracy
        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        accuracy = (
            predictions == y_batch
        ).float().mean().item()

        losses.append(
            loss.item()
        )

        accuracies.append(
            accuracy
        )

    return (
        model,
        history,
        losses,
        accuracies
    )


# ============================================================
# Run all three initialization experiments
# ============================================================

results = {}

for initialization in [
    "uninitialized",
    "xavier",
    "kaiming"
]:

    print(
        f"\nRunning: {initialization.upper()}"
    )

    results[initialization] = run_experiment(
        initialization,
        EPOCHS
    )

print("\nAll experiments completed.")

# Activation Distribution Analysis

The following plots show the mean activation value and activation variance for every layer.

A healthy initialization should generally prevent activation statistics from collapsing toward zero or growing uncontrollably as depth increases.

Xavier and Kaiming initialization are expected to provide more stable activation propagation than an unscaled random initialization.

In [ ]:
# ============================================================
# Activation Mean by Layer
# ============================================================

initializations = [
    "uninitialized",
    "xavier",
    "kaiming"
]

for initialization in initializations:

    history = results[
        initialization
    ][1]

    final_means = [
        history["activation_mean"][layer][-1]
        for layer in range(NUM_LAYERS)
    ]

    plt.figure(figsize=(9, 5))

    plt.plot(
        range(1, NUM_LAYERS + 1),
        final_means,
        marker="o"
    )

    plt.xlabel("Layer")
    plt.ylabel("Mean Activation")
    plt.title(
        f"Activation Mean - {initialization.title()}"
    )

    plt.grid(True)
    plt.show()


# ============================================================
# Activation Variance by Layer
# ============================================================

for initialization in initializations:

    history = results[
        initialization
    ][1]

    final_variances = [
        history[
            "activation_variance"
        ][layer][-1]
        for layer in range(NUM_LAYERS)
    ]

    plt.figure(figsize=(9, 5))

    plt.plot(
        range(1, NUM_LAYERS + 1),
        final_variances,
        marker="o"
    )

    plt.xlabel("Layer")
    plt.ylabel("Activation Variance")
    plt.title(
        f"Activation Variance - {initialization.title()}"
    )

    plt.grid(True)
    plt.show()

# Gradient Magnitude Analysis

The gradient magnitude is analyzed across the 20 layers.

A gradient that becomes progressively smaller toward earlier layers indicates a vanishing-gradient problem.

A gradient that becomes excessively large indicates an exploding-gradient problem.

The plots allow direct comparison of the three initialization strategies.

In [ ]:
# ============================================================
# Gradient Magnitude by Layer
# ============================================================

for initialization in initializations:

    history = results[
        initialization
    ][1]

    final_gradients = [
        history[
            "gradient_mean"
        ][layer][-1]
        for layer in range(NUM_LAYERS)
    ]

    plt.figure(figsize=(9, 5))

    plt.semilogy(
        range(1, NUM_LAYERS + 1),
        np.maximum(
            final_gradients,
            1e-12
        ),
        marker="o"
    )

    plt.xlabel("Layer")
    plt.ylabel("Mean |Gradient|")
    plt.title(
        f"Weight Gradient Magnitude - "
        f"{initialization.title()}"
    )

    plt.grid(True)
    plt.show()


# ============================================================
# Gradient magnitude across epochs
# ============================================================

selected_layers = [
    0,
    4,
    9,
    14,
    19
]

for initialization in initializations:

    history = results[
        initialization
    ][1]

    plt.figure(figsize=(9, 5))

    for layer in selected_layers:

        values = history[
            "gradient_mean"
        ][layer]

        plt.semilogy(
            values,
            label=f"Layer {layer + 1}"
        )

    plt.xlabel("Epoch")
    plt.ylabel("Mean |Gradient|")
    plt.title(
        f"Gradient Magnitude Across Epochs - "
        f"{initialization.title()}"
    )

    plt.legend()
    plt.grid(True)
    plt.show()

# Loss and Accuracy Comparison

The training loss and accuracy are compared for all initialization strategies.

This provides an additional empirical indication of how initialization affects optimization in a deep network.

In [ ]:
# ============================================================
# Loss comparison
# ============================================================

plt.figure(figsize=(9, 5))

for initialization in initializations:

    losses = results[
        initialization
    ][2]

    plt.plot(
        losses,
        label=initialization.title()
    )

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss Comparison")

plt.legend()
plt.grid(True)
plt.show()


# ============================================================
# Accuracy comparison
# ============================================================

plt.figure(figsize=(9, 5))

for initialization in initializations:

    accuracies = results[
        initialization
    ][3]

    plt.plot(
        np.array(accuracies) * 100,
        label=initialization.title()
    )

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training Accuracy Comparison")

plt.legend()
plt.grid(True)
plt.show()

# Results and Conclusion

The experiment investigated the effect of weight initialization on a deep 20-layer MLP.

Three initialization strategies were compared:

- Uninitialized standard-normal weights
- Xavier/Glorot initialization
- Kaiming/He initialization

The diagnostic harness tracked activation means, activation variances, and weight-gradient magnitudes across layers and epochs.

## Expected Observations

### Uninitialized Weights

Using an unscaled normal distribution can cause activation and gradient statistics to become unstable as they propagate through many layers. Depending on the scale of the random weights, this can result in vanishing or exploding values.

### Xavier Initialization

Xavier initialization attempts to maintain a stable variance of activations and gradients across layers. It is generally appropriate for symmetric activation functions and helps reduce training instability.

### Kaiming Initialization

Kaiming initialization is specifically designed for ReLU networks. Because it uses:

\[
Var(W)=\frac{2}{n_{in}}
\]

it generally provides better signal propagation when ReLU activations are used.

## Conclusion

The experiment demonstrates that weight initialization has a significant effect on the behavior of deep neural networks.

Monitoring activation statistics and gradient magnitudes provides an empirical method for diagnosing vanishing and exploding gradients.

The experiment also demonstrates that carefully selected initialization methods such as Xavier and Kaiming can improve the stability of deep-network training compared with an unscaled random initialization.

Most importantly, the diagnostic process does not rely on PyTorch automatic differentiation. The gradients are calculated manually, allowing the relationship between initialization, forward activation propagation, and backward gradient propagation to be directly observed.